<a href="https://colab.research.google.com/github/dipu-malitha/ML-Projects/blob/main/Reaction_Kinetics_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Sonification of Concentration Profiles

In [2]:
# Imports libary
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import wave
import struct
from IPython.display import Audio, display
from google.colab import files

In [3]:
#Reaction Parameters
# ────────────────────────────────────────────────────────────
# Mechanism:
#   E + S  --k1-->  ES       (association)
#   ES     --k-1--> E + S    (dissociation)
#   ES     --k2-->  E + P    (catalysis / kcat)
#
# Parameters based on urease-like enzyme system
# (Cornish-Bowden, Fundamentals of Enzyme Kinetics, 2004)

k1  = 100.0    # M^-1 s^-1   association rate constant
km1 = 0.1      # s^-1        dissociation rate constant
k2  = 0.05     # s^-1        catalytic rate constant (kcat)

S0  = 1.0e-3   # mol/L   initial substrate  (1.0 mM)
E0  = 1.0e-5   # mol/L   initial enzyme     (10 µM)
ES0 = 0.0      # mol/L   no complex initially
P0  = 0.0      # mol/L   no product initially

In [4]:
# Derived quantities
Km   = (km1 + k2) / k1       # Michaelis constant (mol/L)
Vmax = k2 * E0               # Maximum reaction rate (mol/L/s)

print("=" * 50)
print("  MICHAELIS-MENTEN PARAMETERS")
print("=" * 50)
print(f"  k1   = {k1}    M⁻¹s⁻¹")
print(f"  k-1  = {km1}   s⁻¹")
print(f"  k2   = {k2}   s⁻¹  (kcat)")
print(f"  Km   = {Km*1e3:.4f} mM")
print(f"  Vmax = {Vmax*1e9:.2f}  nM/s")
print(f"  [S]0 = {S0*1e3:.2f}   mM")
print(f"  [E]0 = {E0*1e6:.2f}  µM")
print("=" * 50)

  MICHAELIS-MENTEN PARAMETERS
  k1   = 100.0    M⁻¹s⁻¹
  k-1  = 0.1   s⁻¹
  k2   = 0.05   s⁻¹  (kcat)
  Km   = 1.5000 mM
  Vmax = 500.00  nM/s
  [S]0 = 1.00   mM
  [E]0 = 10.00  µM


In [5]:
# CELL 4 — ODE System & Solver (Batch Reactor)

def michaelis_menten_odes(t, y):
    """
    Full mechanistic ODEs for Michaelis-Menten kinetics.
    State vector: y = [S, E, ES, P]
    Reactor model: Isothermal Batch Reactor
    Mole balance: dCi/dt = ri  (no flow terms)
    """
    S, E, ES, P = y
    dS  = -k1*E*S + km1*ES
    dE  = -k1*E*S + km1*ES + k2*ES
    dES =  k1*E*S - km1*ES - k2*ES
    dP  =  k2*ES
    return [dS, dE, dES, dP]

In [6]:
# Time span and evaluation points
t_span = (0, 500)                      # seconds
t_eval = np.linspace(0, 500, 5000)
y0     = [S0, E0, ES0, P0]


In [7]:
# Solve the ODE system
sol = solve_ivp(
    michaelis_menten_odes,
    t_span,
    y0,
    method='RK45',
    t_eval=t_eval,
    rtol=1e-9,
    atol=1e-12
)


In [8]:
# Extract and convert units
t   = sol.t
S   = sol.y[0] * 1e3    # mM
E   = sol.y[1] * 1e6    # µM
ES  = sol.y[2] * 1e6    # µM
P   = sol.y[3] * 1e3    # mM
rxn_rate = k2 * sol.y[2] * 1e9   # nmol/L/s

print (f" rxn_rate = {rxn_rate[-1]:.2f} nmol/L/s")


 rxn_rate = 187.63 nmol/L/s


In [9]:
# Verify enzyme mass conservation
E_total = sol.y[1] + sol.y[2]
deviation = np.max(np.abs(E_total - E0))
print(f"\n✓ Mass conservation check: max deviation = {deviation:.2e} mol/L (should be ~0)")


✓ Mass conservation check: max deviation = 1.69e-21 mol/L (should be ~0)


In [10]:
# CELL 5 — Plot Concentration Profiles
# ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10), facecolor='#0d1117')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

COLORS = {
    'S':    '#00d4ff',
    'E':    '#ff6b6b',
    'ES':   '#ffd93d',
    'P':    '#6bcb77',
    'rate': '#c084fc',
}
def style_ax(ax, title, ylabel, xlabel='Time (s)'):
    ax.set_facecolor('#161b22')
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel, color='#8b949e', fontsize=10)
    ax.set_ylabel(ylabel, color='#8b949e', fontsize=10)
    ax.tick_params(colors='#8b949e')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    ax.grid(True, color='#21262d', linewidth=0.8, linestyle='--')
    return ax

<Figure size 1400x1000 with 0 Axes>

In [11]:
# Panel 1: Substrate & Product
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t, S, color=COLORS['S'], lw=2.2, label='[S] Substrate')
ax1.plot(t, P, color=COLORS['P'], lw=2.2, label='[P] Product')
style_ax(ax1, 'Substrate & Product', 'Concentration (mM)')
ax1.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='white', fontsize=9)

In [12]:
# Panel 2: Enzyme species
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(t, E,  color=COLORS['E'],  lw=2.2, label='[E] Free Enzyme')
ax2.plot(t, ES, color=COLORS['ES'], lw=2.2, label='[ES] Complex')
style_ax(ax2, 'Enzyme & ES Complex', 'Concentration (µM)')
ax2.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='white', fontsize=9)
plt.show()

In [13]:
# Panel 3: All species (dual y-axis)
ax3 = fig.add_subplot(gs[1, 0])
ax3_r = ax3.twinx()
ax3.plot(t, S, color=COLORS['S'], lw=2, label='[S] mM')
ax3.plot(t, P, color=COLORS['P'], lw=2, label='[P] mM')
ax3_r.plot(t, E,  color=COLORS['E'],  lw=2, ls='--', label='[E] µM')
ax3_r.plot(t, ES, color=COLORS['ES'], lw=2, ls='--', label='[ES] µM')
style_ax(ax3, 'All Species (Combined)', 'mM  (S, P)')
ax3_r.set_ylabel('µM  (E, ES)', color='#8b949e', fontsize=10)
ax3_r.tick_params(colors='#8b949e')
ax3_r.set_facecolor('#161b22')
for sp in ax3_r.spines.values(): sp.set_edgecolor('#30363d')
lines = ax3.get_legend_handles_labels()
lines2 = ax3_r.get_legend_handles_labels()
ax3.legend(lines[0]+lines2[0], lines[1]+lines2[1],
           facecolor='#21262d', edgecolor='#30363d', labelcolor='white', fontsize =8)

In [14]:
# Panel 4: Reaction rate
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(t, rxn_rate, color=COLORS['rate'], lw=2.5)
ax4.fill_between(t, rxn_rate, alpha=0.25, color=COLORS['rate'])
style_ax(ax4, 'Reaction Rate  v = k₂·[ES]', 'Rate (nmol/L/s)')

fig.suptitle('Michaelis-Menten Enzyme Kinetics — Batch Reactor\nConcentration Profiles',
             color='white', fontsize=15, fontweight='bold', y=1.01)

plt.savefig('concentration_profiles.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✓ Plot saved: concentration_profiles.png")


<Figure size 640x480 with 0 Axes>

✓ Plot saved: concentration_profiles.png


In [16]:
# CELL 6 — Sonification (WAV Audio)

# SONIFICATION STRATEGY:
#   [S]  (Substrate)    → Pitch of Tone 1   (high conc = high pitch)
#   [P]  (Product)      → Pitch of Tone 2   (lower → higher over time)
#   [ES] (Complex)      → Master amplitude   (rise-peak-fall envelope)
#   [E]  (Free Enzyme)  → Secondary envelope
#   Scale: C Pentatonic Minor (sounds musical for any concentration values)
#   Compression: 500 s reaction → 30 s audio

SAMPLE_RATE = 44100
DURATION    = 30.0
N_SAMPLES   = int(SAMPLE_RATE * DURATION)

In [17]:
# C Pentatonic Minor: MIDI note numbers
SCALE_LOW  = np.array([60, 63, 65, 67, 70, 72])   # C4 – C5
SCALE_HIGH = np.array([72, 75, 77, 79, 82, 84])   # C5 – C6

def midi_to_freq(midi): return 440.0 * (2.0 ** ((midi - 69) / 12.0))

def conc_to_freq(conc_arr, scale):
    """Map concentration array → nearest scale frequency (Hz)."""
    lo, hi = conc_arr.min(), conc_arr.max()
    norm = (conc_arr - lo) / (hi - lo + 1e-12)
    idx  = np.clip((norm * (len(scale) - 1)).astype(int), 0, len(scale)-1)
    return np.array([midi_to_freq(scale[i]) for i in idx])

def smooth(arr, window=2000):
    return np.convolve(arr, np.ones(window)/window, mode='same')

In [18]:
# Re-sample concentrations to audio time grid
t_audio  = np.linspace(0, 500, N_SAMPLES)
S_a  = np.interp(t_audio, t, sol.y[0]) * 1e3
E_a  = np.interp(t_audio, t, sol.y[1]) * 1e6
ES_a = np.interp(t_audio, t, sol.y[2]) * 1e6
P_a  = np.interp(t_audio, t, sol.y[3]) * 1e3

In [19]:
# Frequencies
freq_S = conc_to_freq(S_a,  SCALE_LOW)
freq_P = conc_to_freq(P_a,  SCALE_HIGH)

In [20]:

# Amplitude envelopes (smoothed to avoid clicks)
amp_ES = smooth(ES_a / (ES_a.max() + 1e-12))
amp_E  = smooth(E_a  / (E_a.max()  + 1e-12))

# Synthesize audio — phase-continuous to prevent discontinuities
time_vec = np.arange(N_SAMPLES) / SAMPLE_RATE

In [21]:
# Tone 1: Substrate voice + slow vibrato (4 Hz)
phase1  = np.cumsum(2 * np.pi * freq_S / SAMPLE_RATE)
vibrato = 0.003 * np.sin(2 * np.pi * 4 * time_vec)
tone1   = amp_ES * np.sin(phase1 + vibrato)

# Tone 2: Product voice (higher register, softer)
phase2 = np.cumsum(2 * np.pi * freq_P / SAMPLE_RATE)
tone2  = amp_E * 0.6 * np.sin(phase2)

# Tone 3: ES complex pulse — adds harmonic texture at the peak
pulse_freq  = 2.0 + 4.0 * amp_ES
pulse_phase = np.cumsum(2 * np.pi * pulse_freq / SAMPLE_RATE)
tone3 = amp_ES * 0.3 * np.sin(pulse_phase) * np.sin(phase1 * 2)


In [22]:
# Mix and normalize
audio = tone1 + tone2 + tone3
audio = audio / (np.max(np.abs(audio)) + 1e-12) * 0.85
audio_int16 = (audio * 32767).astype(np.int16)

In [23]:
# Write WAV file
wav_path = 'michaelis_menten_sonification.wav'
with wave.open(wav_path, 'w') as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(SAMPLE_RATE)
    wf.writeframes(audio_int16.tobytes())

print(f"✓ WAV saved: {wav_path}  ({DURATION:.0f} s, {SAMPLE_RATE} Hz, 16-bit mono)")

# Play audio directly in Colab
print("\n🎵 Playing sonification in notebook...")
display(Audio(audio, rate=SAMPLE_RATE))

✓ WAV saved: michaelis_menten_sonification.wav  (30 s, 44100 Hz, 16-bit mono)

🎵 Playing sonification in notebook...


In [25]:
#CELL 8 — Download All Files
print("⬇️  Downloading files to your computer...")
files.download('concentration_profiles.png')
files.download('michaelis_menten_sonification.wav')
print("✓ Done! Check your Downloads folder.")

⬇️  Downloading files to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Done! Check your Downloads folder.
